# Session 6 — Analysis: Framework Interoperability (JAX/Flax and PyTorch/XLA on shared hardware)

## Background

Sessions 1–5 benchmarked PyTorch (eager GPU) and PyTorch/XLA (TPU). Session 6 ran the same Transformer architecture — same model size, same batch sweep, same train step structure — using JAX/Flax instead. Both JAX and PyTorch/XLA compile to XLA HLO before executing on the TPU. In principle, the generated HLO should be nearly identical, which means throughput should converge. On GPU, native PyTorch's cuDNN/cuBLAS backend is more mature than JAX's CUDA XLA path, so PyTorch is expected to outperform JAX.

This notebook tests those predictions, quantifies how much (if anything) framework choice costs in throughput, and measures the compilation overhead of each path.

Two production ML frameworks that both compile to XLA HLO on shared hardware make different tradeoffs:
- **Graph granularity:** JAX jit-compiles the entire train step; PyTorch/XLA traces lazily and compiles at `mark_step()`
- **Compile strategy:** JAX compiles eagerly on first call per shape; PyTorch/XLA amortizes over warmup
- **Batch-size scaling:** As this session shows, the scaling behaviour diverges — especially on TPU

## Goal

- Overlay JAX and PyTorch throughput curves per device
- Compare first-step compilation times
- Reframe the "framework choice" question as a throughput question vs an ergonomics question

## Question

Does switching from PyTorch/XLA to JAX/Flax change TPU throughput? Does native PyTorch outperform JAX on GPU, and by how much?

---

**Prerequisites:** Run `01-benchmark-gpu.ipynb` and `02-benchmark-tpu.ipynb` first, then `../session_1/` if Session 1 results are not already present.

In [ ]:
import json, os

def load_json(path):
    if not os.path.exists(path):
        print(f"  Missing: {path}")
        return None
    with open(path) as f:
        return json.load(f)

# Session 6 JAX results
gpu_jax = load_json("results/gpu_jax.json")
tpu_jax = load_json("results/tpu_jax.json")

# Session 1 PyTorch baselines (for overlay)
gpu_pt  = load_json("../session_1/results/gpu.json")
tpu_pt  = load_json("../session_1/results/tpu.json")

for label, d in [("GPU JAX", gpu_jax), ("TPU JAX", tpu_jax),
                  ("GPU PyTorch (S1)", gpu_pt), ("TPU PyTorch/XLA (S1)", tpu_pt)]:
    if d:
        hw  = d.get("hardware", "?")
        fw  = d.get("framework", "PyTorch")
        ts  = d.get("timestamp", "unknown")
        print(f"  {label:<26}: {hw} / {fw}  @  {ts}")
    else:
        print(f"  {label:<26}: NOT LOADED")

  GPU JAX                   : GPU / JAX/Flax  @  2026-02-28T00:38:16.882987+00:00
  TPU JAX                   : TPU / JAX/Flax  @  2026-02-28T00:36:50.890950+00:00
  GPU PyTorch (S1)          : GPU / PyTorch  @  2026-02-27T04:53:26.664244
  TPU PyTorch/XLA (S1)      : TPU / PyTorch  @  2026-02-26T18:52:53.393683


In [ ]:
BATCH_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]

def get_tput(d, bs):
    """Extract throughput from either JAX results (nested) or PyTorch results (flat)."""
    if d is None:
        return None
    r = d.get("results", {}).get(str(bs))
    if r is None:
        return None
    if isinstance(r, dict):
        return r.get("throughput")
    return r

print(f"{'─'*72}")
print(f"  GPU throughput (samples/sec)")
print(f"  {'batch':<8}  {'JAX/Flax':>14}  {'PyTorch (S1)':>14}  {'PT/JAX ratio':>14}")
print(f"  {'─'*8}  {'─'*14}  {'─'*14}  {'─'*14}")
for bs in BATCH_SIZES:
    jt = get_tput(gpu_jax, bs)
    pt = get_tput(gpu_pt, bs)
    ratio = f"{pt/jt:.3f}×" if jt and pt else "—"
    print(f"  {bs:<8}  {jt:>14,.0f}  {pt:>14,.0f}  {ratio:>14}" if jt and pt else
          f"  {bs:<8}  {'—':>14}  {'—':>14}  {'—':>14}")

print(f"\n{'─'*72}")
print(f"  TPU throughput (samples/sec)")
print(f"  {'batch':<8}  {'JAX/Flax':>14}  {'PyTorch/XLA (S1)':>16}  {'JAX/PT ratio':>14}")
print(f"  {'─'*8}  {'─'*14}  {'─'*16}  {'─'*14}")
for bs in BATCH_SIZES:
    jt = get_tput(tpu_jax, bs)
    pt = get_tput(tpu_pt, bs)
    ratio = f"{jt/pt:.3f}×" if jt and pt else "—"
    print(f"  {bs:<8}  {jt:>14,.0f}  {pt:>16,.0f}  {ratio:>14}" if jt and pt else
          f"  {bs:<8}  {'—':>14}  {'—':>16}  {'—':>14}")

────────────────────────────────────────────────────────────────────────
  GPU throughput (samples/sec)
  batch           JAX/Flax    PyTorch (S1)    PT/JAX ratio
  ────────  ──────────────  ──────────────  ──────────────
  4                  3,186           1,388          0.436×
  8                  4,409           2,521          0.572×
  16                 5,946           2,866          0.482×
  32                 6,070           2,895          0.477×
  64                 5,304           2,760          0.520×
  128                4,823           2,688          0.557×
  256                4,660           2,653          0.569×
  512                4,571           2,618          0.573×
  1024               4,463           2,575          0.577×

────────────────────────────────────────────────────────────────────────
  TPU throughput (samples/sec)
  batch           JAX/Flax  PyTorch/XLA (S1)    JAX/PT ratio
  ────────  ──────────────  ────────────────  ──────────────
  4                 

In [ ]:
print("Compilation times — JAX (first call per batch shape)")
print(f"{'─'*50}")
print(f"  {'batch':<8}  {'GPU JAX':>12}  {'TPU JAX':>12}")
print(f"  {'─'*8}  {'─'*12}  {'─'*12}")

gpu_ct = gpu_jax.get("compile_times_s", {}) if gpu_jax else {}
tpu_ct = tpu_jax.get("compile_times_s", {}) if tpu_jax else {}

for bs in BATCH_SIZES:
    gct = gpu_ct.get(str(bs))
    tct = tpu_ct.get(str(bs))
    gs  = f"{gct:.2f} s" if gct is not None else "—"
    ts  = f"{tct:.2f} s" if tct is not None else "—"
    print(f"  {bs:<8}  {gs:>12}  {ts:>12}")

print()
print("Note: PyTorch GPU has no compilation step (eager execution).")
print("PyTorch/XLA TPU compilation cost is absorbed by Session 1's warmup steps.")

Compilation times — JAX (first call per batch shape)
──────────────────────────────────────────────────
  batch          GPU JAX       TPU JAX
  ────────  ────────────  ────────────
  4              15.63 s        2.47 s
  8              18.02 s        3.55 s
  16             19.52 s        3.83 s
  32             22.55 s        3.87 s
  64             18.81 s        2.77 s
  128            19.55 s        3.27 s
  256            20.43 s        3.28 s
  512            22.04 s        3.46 s
  1024           26.53 s        3.47 s

Note: PyTorch GPU has no compilation step (eager execution).
PyTorch/XLA TPU compilation cost is absorbed by Session 1's warmup steps.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

BATCH_SIZES_PLOT = [bs for bs in BATCH_SIZES
                    if get_tput(gpu_jax, bs) or get_tput(gpu_pt, bs)
                       or get_tput(tpu_jax, bs) or get_tput(tpu_pt, bs)]

COLORS = {
    "gpu_jax": "#ff9900",
    "gpu_pt":  "#76b900",
    "tpu_jax": "#4285F4",
    "tpu_pt":  "#34A853",
}

# ── Chart 1: Throughput vs batch size ─────────────────────────────────────────
fig, (ax_gpu, ax_tpu) = plt.subplots(1, 2, figsize=(14, 5))

for ax, (jax_d, pt_d, jax_key, pt_key, jax_label, pt_label, title) in [
    (ax_gpu, (gpu_jax, gpu_pt, "gpu_jax", "gpu_pt",
              "GPU — JAX/Flax", "GPU — PyTorch (S1)", "GPU Throughput")),
    (ax_tpu, (tpu_jax, tpu_pt, "tpu_jax", "tpu_pt",
              "TPU — JAX/Flax", "TPU — PyTorch/XLA (S1)", "TPU Throughput")),
]:
    for d, key, label in [(jax_d, jax_key, jax_label), (pt_d, pt_key, pt_label)]:
        xs = [bs for bs in BATCH_SIZES_PLOT if get_tput(d, bs) is not None]
        ys = [get_tput(d, bs) for bs in xs]
        if xs:
            ax.plot(xs, ys, marker="o", label=label,
                    color=COLORS[key], linewidth=2, markersize=5)
    ax.set_xscale("log", base=2)
    ax.set_xticks(BATCH_SIZES_PLOT)
    ax.set_xticklabels(BATCH_SIZES_PLOT)
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel("Throughput (samples / second)", fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.suptitle("Session 6 — JAX/Flax vs PyTorch(/XLA) Throughput", fontsize=13)
plt.tight_layout()
os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_6_chart_throughput.png", dpi=150)
print("Saved session_6_chart_throughput.png")

Saved session_6_chart_throughput.png


In [ ]:
# ── Chart 2: JAX/PyTorch throughput ratio per device ─────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))

for d_jax, d_pt, color, label in [
    (gpu_jax, gpu_pt, COLORS["gpu_jax"], "GPU (JAX / PyTorch)"),
    (tpu_jax, tpu_pt, COLORS["tpu_jax"], "TPU (JAX / PyTorch/XLA)"),
]:
    xs, ys = [], []
    for bs in BATCH_SIZES_PLOT:
        jt = get_tput(d_jax, bs)
        pt = get_tput(d_pt, bs)
        if jt and pt:
            xs.append(bs)
            ys.append(jt / pt)
    if xs:
        ax2.plot(xs, ys, marker="o", label=label, color=color, linewidth=2)

ax2.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="Parity (1.0×)")
ax2.set_xscale("log", base=2)
ax2.set_xticks(BATCH_SIZES_PLOT)
ax2.set_xticklabels(BATCH_SIZES_PLOT)
ax2.set_xlabel("Batch size", fontsize=11)
ax2.set_ylabel("JAX / PyTorch throughput ratio", fontsize=11)
ax2.set_title("JAX vs PyTorch — Relative Throughput per Device", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("../docs/assets/session_6_chart_ratio.png", dpi=150)
print("Saved session_6_chart_ratio.png")

Saved session_6_chart_ratio.png


In [ ]:
# ── Chart 3: Compilation time per batch size ──────────────────────────────────
fig3, ax3 = plt.subplots(figsize=(8, 4))

for ct_dict, color, label in [
    (gpu_ct, COLORS["gpu_jax"], "GPU — JAX/Flax"),
    (tpu_ct, COLORS["tpu_jax"], "TPU — JAX/Flax"),
]:
    xs = [int(bs) for bs in ct_dict if ct_dict[bs] is not None]
    ys = [ct_dict[str(bs)] for bs in xs if ct_dict.get(str(bs)) is not None]
    xs = [bs for bs in xs if ct_dict.get(str(bs)) is not None]
    if xs:
        ax3.plot(xs, ys, marker="o", label=label, color=color, linewidth=2)

ax3.set_xscale("log", base=2)
ax3.set_xticks(BATCH_SIZES_PLOT)
ax3.set_xticklabels(BATCH_SIZES_PLOT)
ax3.set_xlabel("Batch size", fontsize=11)
ax3.set_ylabel("First-call compilation time (s)", fontsize=11)
ax3.set_title("JAX XLA Compilation Time per Input Shape", fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("../docs/assets/session_6_chart_compile.png", dpi=150)
print("Saved session_6_chart_compile.png")

Saved session_6_chart_compile.png


## Interpreting the results

### TPU: framework convergence

JAX/Flax and PyTorch/XLA both compile to XLA HLO before executing on the TPU's MXU. If the JAX/PyTorch ratio on TPU is close to 1.0×, it confirms the principle: **the framework choice on TPU is not a throughput decision**. Any difference reflects graph fusion quality in the respective front-ends (how well each framework constructs the HLO graph), not a fundamental architecture difference.

### GPU: PyTorch advantage expected

On GPU, native PyTorch dispatches directly to cuBLAS/cuDNN — highly tuned NVIDIA libraries with operator-specific kernels optimized over many years. JAX's CUDA XLA backend routes through XLA's CUDA codegen path, which is general-purpose and lacks some of the cuDNN specializations. A GPU JAX/PyTorch ratio below 1.0× is therefore expected.

### Compilation cost

JAX compiles on the first call for each unique input shape. The compilation time chart shows how this cost varies with batch size. Unlike PyTorch eager (no compilation cost), JAX GPU also pays this cost — though the steady-state throughput may still trail PyTorch. On TPU, this is the same compilation spike that PyTorch/XLA absorbs in its warmup steps.

### Decision guidance

| Scenario | Recommendation |
|---|---|
| Existing PyTorch code, targeting TPU | Use PyTorch/XLA. Near-identical throughput, minimal migration cost. |
| New project, large-scale TPU training | Consider JAX. Functional API, simpler multi-device parallelism, same throughput. |
| Existing PyTorch code, staying on GPU | Stay on PyTorch. JAX GPU trails on throughput, no benefit to migrate. |
| JAX research codebase, adding GPU runs | JAX is acceptable on GPU; expect some throughput penalty vs PyTorch native. |